In [8]:
from src.envs import N3il

In [9]:
n = 5 # Example grid size, can be adjusted as needed
i = 0 # Random seed index, can be adjusted as needed

args = {
    'algorithm': 'MCTS',
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 10*(n**2),  # Adjusted for larger n
    'num_workers': 28,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': False,
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': f'tests/tests_mcts',  # Directory to save tables
    'figure_dir': f'tests/tests_mcts/figure',  # Directory to save figures
    'random_seed': i,  # Use the loop index as a seed for reproducibility
}

n3il = N3il((n,n), args=args)

In [15]:
state = n3il.get_initial_state()

In [16]:
state

array([[0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [ ]:
n3il.display_state()

In [22]:
import numpy as np
from numba import njit

class KillSpaceAnalyzer(N3il):
    """
    Inherits from N3il and adds functionality to analyze how many valid spots 
    each potential move would kill.
    """
    
    def __init__(self, grid_size, args, priority_grid=None):
        super().__init__(grid_size, args, priority_grid)
    
    def get_killed_action_space_priority(self, state):
        """
        Given a state, for each valid action spot, calculate how many valid spots 
        would be killed if a point is added to that spot.
        
        Returns a 2D grid where:
        - For valid moves: -k (where k is the number of spots that would be killed)
        - For invalid moves: -inf
        
        The goal is to argmax this grid to get the best next move.
        """
        rows, cols = self.row_count, self.column_count
        
        # Get current valid moves
        current_valid_moves = self.get_valid_moves_nb_raw(state, rows, cols)
        current_valid_count = int(np.sum(current_valid_moves))  # Cast to int to avoid uint64 issues
        
        # Initialize priority grid with -inf for invalid moves
        priority_grid = np.full((rows, cols), -np.inf, dtype=np.float64)
        
        # For each possible action spot
        for i in range(rows):
            for j in range(cols):
                action_idx = i * cols + j
                
                # Skip if current spot is occupied
                if state[i, j] == 1:
                    continue
                    
                # Skip if this is not a valid move
                if current_valid_moves[action_idx] == 0:
                    continue
                
                # Calculate new valid moves after this action
                new_valid_moves = self.get_valid_moves_subset_nb_raw(
                    state, current_valid_moves, action_idx, rows, cols
                )
                new_valid_count = int(np.sum(new_valid_moves))  # Cast to int to avoid uint64 issues
                
                # Calculate how many spots were killed
                # We subtract 1 because we're placing one point (the action itself)
                killed_count = current_valid_count - new_valid_count - 1
                
                # Assign negative killed count as priority (now safe with signed integers)
                priority_grid[i, j] = -killed_count
        
        return priority_grid
    
    def get_valid_moves_nb_raw(self, state, row_count, column_count):
        """Raw version of get_valid_moves_nb without priority filtering"""
        return get_valid_moves_nb_raw(state, row_count, column_count)
    
    def get_valid_moves_subset_nb_raw(self, parent_state, parent_valid_moves, action_taken, row_count, column_count):
        """Raw version of get_valid_moves_subset_nb without priority filtering"""
        return get_valid_moves_subset_nb_raw(parent_state, parent_valid_moves, action_taken, row_count, column_count)
    
    def get_best_action(self, state):
        """
        Get the best action by finding the argmax of the killed action space priority grid.
        Returns the (row, col) coordinates of the best move.
        """
        priority_grid = self.get_killed_action_space_priority(state)
        
        # Check if any finite values exist (valid moves)
        if not np.any(np.isfinite(priority_grid)):
            return None
        
        # Find the position with maximum priority (least negative = kills fewest spots)
        max_pos = np.unravel_index(np.argmax(priority_grid), priority_grid.shape)
        
        return max_pos


# Define the raw numba functions (copies from the main file but without priority filtering)
@njit(cache=True, nogil=True)
def _are_collinear_raw(x1, y1, x2, y2, x3, y3):
    """Check if three points are collinear"""
    return (y1 - y2) * (x1 - x3) == (y1 - y3) * (x1 - x2)

@njit(cache=True, nogil=True)
def get_valid_moves_nb_raw(state, row_count, column_count):
    """Raw version of get_valid_moves_nb without any priority filtering"""
    max_pts = row_count * column_count
    coords = np.empty((max_pts, 2), np.int64)
    n_pts = 0

    # Collect coordinates of existing points
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] == 1:
                coords[n_pts, 0] = i
                coords[n_pts, 1] = j
                n_pts += 1

    mask = np.zeros(row_count * column_count, np.uint8)

    # Check each empty cell
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] != 0:
                continue
            valid = True
            # Check for collinearity with every pair of existing points
            for p in range(n_pts):
                for q in range(p + 1, n_pts):
                    i1, j1 = coords[p, 0], coords[p, 1]
                    i2, j2 = coords[q, 0], coords[q, 1]
                    if _are_collinear_raw(j1, i1, j2, i2, j, i):
                        valid = False
                        break
                if not valid:
                    break
            if valid:
                mask[i * column_count + j] = 1
    return mask

@njit(cache=True, nogil=True)
def get_valid_moves_subset_nb_raw(parent_state, parent_valid_moves, action_taken, row_count, column_count):
    """Raw version of get_valid_moves_subset_nb without any priority filtering"""
    # Copy input mask and remove the taken action
    mask = parent_valid_moves.copy()
    mask[action_taken] = 0

    # Coordinates of the newly placed point
    new_r = action_taken // column_count
    new_c = action_taken % column_count

    # Iterate over all existing points
    for pr in range(row_count):
        for pc in range(column_count):
            if not parent_state[pr, pc]:
                continue
            # Skip the new point itself
            if pr == new_r and pc == new_c:
                continue

            dr = pr - new_r
            dc = pc - new_c

            # Infinite slope (vertical line): invalidate entire column
            if dc == 0:
                for rr in range(row_count):
                    idx = rr * column_count + new_c
                    mask[idx] = 0
                continue

            # Zero slope (horizontal line): invalidate entire row
            if dr == 0:
                row_index = pr
                base = row_index * column_count
                for cc in range(column_count):
                    mask[base + cc] = 0
                continue

            # General case: invalidate points on the line
            for cc in range(column_count):
                num = (cc - new_c) * dr
                if num % dc != 0:
                    continue
                rr = new_r + num // dc
                if rr < 0 or rr >= row_count:
                    continue
                idx = rr * column_count + cc
                mask[idx] = 0

    return mask

In [23]:
# Test the example from the user
# Create a KillSpaceAnalyzer for 3x3 grid
analyzer_3x3 = KillSpaceAnalyzer((3, 3), args=args)

# Create the test state:
# 0 1 1
# 0 0 0  
# 0 0 0
test_state = np.array([
    [0, 1, 1],
    [0, 0, 0],
    [0, 0, 0]
], dtype=np.uint8)

print("Test state:")
print(test_state)
print()

# Get the killed action space priority
priority_grid = analyzer_3x3.get_killed_action_space_priority(test_state)

print("Priority grid (number of spots killed with negative sign):")
print(priority_grid)
print()

# Display current valid moves for reference
current_valid_moves = analyzer_3x3.get_valid_moves_nb_raw(test_state, 3, 3)
valid_2d = current_valid_moves.reshape(3, 3)
print("Current valid moves (1=valid, 0=invalid):")
print(valid_2d)
print()

# Let's verify by manually checking some positions
print("Manual verification:")
print("Current valid count:", np.sum(current_valid_moves))

# Test placing at position (1, 2) - second row, last column
temp_state = test_state.copy()
temp_state[1, 2] = 1
action_idx = 1 * 3 + 2  # = 5
new_valid_moves = analyzer_3x3.get_valid_moves_subset_nb_raw(test_state, current_valid_moves, action_idx, 3, 3)
print(f"After placing at (1,2): valid count = {np.sum(new_valid_moves)}, killed = {np.sum(current_valid_moves) - np.sum(new_valid_moves) - 1}")

# Test placing at position (1, 1) - second row, second column  
temp_state = test_state.copy()
temp_state[1, 1] = 1
action_idx = 1 * 3 + 1  # = 4
new_valid_moves = analyzer_3x3.get_valid_moves_subset_nb_raw(test_state, current_valid_moves, action_idx, 3, 3)
print(f"After placing at (1,1): valid count = {np.sum(new_valid_moves)}, killed = {np.sum(current_valid_moves) - np.sum(new_valid_moves) - 1}")

# Get best action
best_action = analyzer_3x3.get_best_action(test_state)
print(f"Best action (kills fewest spots): {best_action}")

Test state:
[[0 1 1]
 [0 0 0]
 [0 0 0]]

Priority grid (number of spots killed with negative sign):
[[-inf -inf -inf]
 [  0.  -2.  -1.]
 [ -1.  -1.  -1.]]

Current valid moves (1=valid, 0=invalid):
[[0 0 0]
 [1 1 1]
 [1 1 1]]

Manual verification:
Current valid count: 6
After placing at (1,2): valid count = 4, killed = 1
After placing at (1,1): valid count = 3, killed = 2
Best action (kills fewest spots): (np.int64(1), np.int64(0))


In [24]:
# Create a KillSpaceAnalyzer for 5x5 grid
analyzer_5x5 = KillSpaceAnalyzer((5, 5), args=args)

# Start with initial empty state
state_5x5 = analyzer_5x5.get_initial_state()

print("=== 5x5 Grid Example ===")
print("Starting with empty 5x5 grid")
print()

# Simulate a few moves using the analyzer
for move_num in range(1, 6):
    print(f"--- Move {move_num} ---")
    
    # Get current valid moves count
    current_valid = analyzer_5x5.get_valid_moves_nb_raw(state_5x5, 5, 5)
    print(f"Current valid spots: {np.sum(current_valid)}")
    
    # Get priority grid
    priority_grid = analyzer_5x5.get_killed_action_space_priority(state_5x5)
    
    # Display the priority grid
    print("Priority grid (negative spots killed):")
    with np.printoptions(precision=0, suppress=True):
        print(priority_grid)
    print()
    
    # Get best action (kills fewest spots)
    best_action = analyzer_5x5.get_best_action(state_5x5)
    print(f"Best action (row, col): {best_action}")
    print(f"Priority at best action: {priority_grid[best_action]}")
    
    # Apply the best action
    state_5x5[best_action] = 1
    print(f"Placed point at {best_action}")
    
    # Display current state
    print("Current state:")
    print(state_5x5)
    print()
    
print("=== Final State Analysis ===")
# Final analysis
final_valid = analyzer_5x5.get_valid_moves_nb_raw(state_5x5, 5, 5)
final_priority = analyzer_5x5.get_killed_action_space_priority(state_5x5)

print(f"Final valid spots remaining: {np.sum(final_valid)}")
print("Final priority grid:")
with np.printoptions(precision=0, suppress=True):
    print(final_priority)

# Use the enhanced display method to visualize
print("\\nVisualizing final state:")
analyzer_5x5.display_state(state_5x5)

=== 5x5 Grid Example ===
Starting with empty 5x5 grid

--- Move 1 ---
Current valid spots: 25
Priority grid (negative spots killed):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Best action (row, col): (np.int64(0), np.int64(0))
Priority at best action: 0.0
Placed point at (np.int64(0), np.int64(0))
Current state:
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

--- Move 2 ---
Current valid spots: 24
Priority grid (negative spots killed):
[[-inf  -3.  -3.  -3.  -3.]
 [ -3.  -3.  -1.   0.   0.]
 [ -3.  -1.  -3.   0.  -1.]
 [ -3.   0.   0.  -3.   0.]
 [ -3.   0.  -1.   0.  -3.]]

Best action (row, col): (np.int64(1), np.int64(3))
Priority at best action: 0.0
Placed point at (np.int64(1), np.int64(3))
Current state:
[[1 0 0 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

--- Move 3 ---
Current valid spots: 23
Priority grid (negative spots killed):
[[-inf  -3.  -4.  -6.  -6.]
 [ -6.  -6.  -4. -inf  -3.]
 [ -3.  -1.

In [25]:
# Demonstrate the argmax functionality more clearly
class BestMoveSelector(KillSpaceAnalyzer):
    """
    Extended class that provides clean interface for best move selection
    """
    
    def select_best_move(self, state, verbose=True):
        """
        Select the best move that kills the fewest valid spots.
        
        Args:
            state: Current game state
            verbose: Whether to print detailed information
            
        Returns:
            tuple: (row, col) of best move, or None if no valid moves
        """
        priority_grid = self.get_killed_action_space_priority(state)
        
        # Check if any valid moves exist
        if np.all(np.isinf(priority_grid)):
            if verbose:
                print("No valid moves available!")
            return None
        
        # Find the best move (maximum priority = kills fewest spots)
        best_position = np.unravel_index(np.argmax(priority_grid), priority_grid.shape)
        best_priority = priority_grid[best_position]
        
        if verbose:
            print(f"Best move: {best_position}")
            print(f"Will kill {-int(best_priority)} valid spots")
            print("Priority grid:")
            with np.printoptions(precision=0, suppress=True):
                print(priority_grid)
        
        return best_position
    
    def play_optimal_game(self, max_moves=10):
        """
        Play a game using the optimal strategy (kill fewest spots each turn)
        
        Args:
            max_moves: Maximum number of moves to play
        """
        state = self.get_initial_state()
        move_count = 0
        
        print(f"=== Playing Optimal Game on {self.row_count}x{self.column_count} Grid ===")
        print("Strategy: Always choose move that kills fewest valid spots\\n")
        
        while move_count < max_moves:
            print(f"--- Move {move_count + 1} ---")
            
            # Get current state info
            valid_moves = self.get_valid_moves_nb_raw(state, self.row_count, self.column_count)
            print(f"Valid spots available: {np.sum(valid_moves)}")
            
            # Select best move
            best_move = self.select_best_move(state, verbose=True)
            
            if best_move is None:
                print("Game over - no valid moves!")
                break
            
            # Apply the move
            state[best_move] = 1
            move_count += 1
            
            print(f"Applied move at {best_move}")
            print("Updated state:")
            print(state)
            print()
        
        print(f"Game completed after {move_count} moves")
        print(f"Final points placed: {np.sum(state)}")
        
        return state


# Test the enhanced selector
print("=== Testing BestMoveSelector ===")
selector = BestMoveSelector((4, 4), args=args)

# Test on a small state first
small_state = np.array([
    [0, 1, 0, 0],
    [0, 0, 1, 0], 
    [0, 0, 0, 0],
    [0, 0, 0, 0]
], dtype=np.uint8)

print("Test state (4x4):")
print(small_state)
print()

best_move = selector.select_best_move(small_state)
print(f"\\nRecommended next move: {best_move}")
print()

# Play a short optimal game
final_state = selector.play_optimal_game(max_moves=5)

=== Testing BestMoveSelector ===
Test state (4x4):
[[0 1 0 0]
 [0 0 1 0]
 [0 0 0 0]
 [0 0 0 0]]

Best move: (np.int64(2), np.int64(0))
Will kill 0 valid spots
Priority grid:
[[ -2. -inf  -4.  -4.]
 [ -2.  -4. -inf  -2.]
 [  0.  -4.  -2. -inf]
 [ -2.  -2.  -2.   0.]]
\nRecommended next move: (np.int64(2), np.int64(0))

=== Playing Optimal Game on 4x4 Grid ===
Strategy: Always choose move that kills fewest valid spots\n
--- Move 1 ---
Valid spots available: 16
Best move: (np.int64(0), np.int64(0))
Will kill 0 valid spots
Priority grid:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Applied move at (np.int64(0), np.int64(0))
Updated state:
[[1 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]

--- Move 2 ---
Valid spots available: 15
Best move: (np.int64(1), np.int64(2))
Will kill 0 valid spots
Priority grid:
[[-inf  -2.  -2.  -2.]
 [ -2.  -2.   0.   0.]
 [ -2.   0.  -2.   0.]
 [ -2.   0.   0.  -2.]]
Applied move at (np.int64(1), np.int64(2))
Updated state:
[[1 0 0 0]
 [0 0 1 0]
 [0 

In [21]:
# Debug the overflow issue
print("=== Debugging Overflow Issue ===")

# Create the test state again
test_state = np.array([
    [0, 1, 1],
    [0, 0, 0],
    [0, 0, 0]
], dtype=np.uint8)

analyzer_debug = KillSpaceAnalyzer((3, 3), args=args)

# Get current valid moves
current_valid_moves = analyzer_debug.get_valid_moves_nb_raw(test_state, 3, 3)
current_valid_count = np.sum(current_valid_moves)

print(f"Current valid count: {current_valid_count}")
print(f"Current valid moves: {current_valid_moves}")

# Test specific positions manually
for i in range(3):
    for j in range(3):
        action_idx = i * 3 + j
        
        if test_state[i, j] == 1:
            print(f"Position ({i},{j}): occupied")
            continue
            
        if current_valid_moves[action_idx] == 0:
            print(f"Position ({i},{j}): invalid move")
            continue
        
        print(f"\\nTesting position ({i},{j}):")
        print(f"  Action index: {action_idx}")
        
        # Calculate new valid moves after this action
        new_valid_moves = analyzer_debug.get_valid_moves_subset_nb_raw(
            test_state, current_valid_moves, action_idx, 3, 3
        )
        new_valid_count = np.sum(new_valid_moves)
        
        print(f"  New valid count: {new_valid_count}")
        print(f"  Killed count: {current_valid_count - new_valid_count - 1}")
        
        # Check for any huge numbers in the calculation
        killed_count = current_valid_count - new_valid_count - 1
        print(f"  Priority value: {-killed_count}")
        print(f"  Type: {type(killed_count)}, Value: {killed_count}")
        
        if abs(killed_count) > 1000000:
            print(f"  WARNING: Very large killed count detected!")
            print(f"  current_valid_count: {current_valid_count} (type: {type(current_valid_count)})")
            print(f"  new_valid_count: {new_valid_count} (type: {type(new_valid_count)})")
            print(f"  difference: {current_valid_count - new_valid_count}")
            print(f"  calculation: {current_valid_count} - {new_valid_count} - 1 = {killed_count}")

=== Debugging Overflow Issue ===
Current valid count: 6
Current valid moves: [0 0 0 1 1 1 1 1 1]
Position (0,0): invalid move
Position (0,1): occupied
Position (0,2): occupied
\nTesting position (1,0):
  Action index: 3
  New valid count: 5
  Killed count: 0
  Priority value: 0
  Type: <class 'numpy.uint64'>, Value: 0
\nTesting position (1,1):
  Action index: 4
  New valid count: 3
  Killed count: 2
  Priority value: 18446744073709551614
  Type: <class 'numpy.uint64'>, Value: 2
\nTesting position (1,2):
  Action index: 5
  New valid count: 4
  Killed count: 1
  Priority value: 18446744073709551615
  Type: <class 'numpy.uint64'>, Value: 1
\nTesting position (2,0):
  Action index: 6
  New valid count: 4
  Killed count: 1
  Priority value: 18446744073709551615
  Type: <class 'numpy.uint64'>, Value: 1
\nTesting position (2,1):
  Action index: 7
  New valid count: 4
  Killed count: 1
  Priority value: 18446744073709551615
  Type: <class 'numpy.uint64'>, Value: 1
\nTesting position (2,2):
  

/var/folders/rf/j2243n211434v9h6hz0cc8fc0000gn/T/ipykernel_24053/3849872306.py:47: RuntimeWarning: overflow encountered in scalar negative
  print(f"  Priority value: {-killed_count}")


In [26]:
# === SUMMARY: Implementation Complete ===
print("🎯 IMPLEMENTATION SUMMARY")
print("=" * 50)
print()

print("✅ Successfully implemented KillSpaceAnalyzer class that inherits from N3il")
print("✅ Added get_killed_action_space_priority() method")
print("✅ Added get_best_action() method with argmax functionality")
print("✅ Added BestMoveSelector class for clean interface")
print()

print("📊 KEY FEATURES:")
print("- Calculates how many valid spots each move would kill")
print("- Returns priority grid: -k for valid moves (k = spots killed), -inf for invalid")
print("- argmax gives best move (kills fewest spots)")
print("- Efficient implementation using numba-compiled functions")
print()

print("🧪 VERIFIED EXAMPLES:")
print("1. 3x3 test case matches expected pattern")
print("2. 5x5 automated gameplay works correctly")  
print("3. BestMoveSelector provides clean interface")
print()

print("🎮 USAGE EXAMPLE:")
print("analyzer = KillSpaceAnalyzer((n, n), args)")
print("priority_grid = analyzer.get_killed_action_space_priority(state)")
print("best_move = analyzer.get_best_action(state)")
print("# OR use the enhanced selector:")
print("selector = BestMoveSelector((n, n), args)")
print("best_move = selector.select_best_move(state)")
print()

print("✨ The implementation is ready to use!")

# Quick final test with a simple 3x3 case
print("\\n" + "="*50)
print("🔬 FINAL VALIDATION")
simple_state = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=np.uint8)
analyzer_final = KillSpaceAnalyzer((3, 3), args=args)
priority = analyzer_final.get_killed_action_space_priority(simple_state)

print("Simple diagonal state:")
print(simple_state)
print("\\nPriority grid:")
with np.printoptions(precision=1, suppress=True):
    print(priority)
    
best = analyzer_final.get_best_action(simple_state)
print(f"\\nBest move: {best}")
print(f"Priority at best move: {priority[best]:.0f}")
print("\\n✅ All systems working correctly!")

🎯 IMPLEMENTATION SUMMARY

✅ Successfully implemented KillSpaceAnalyzer class that inherits from N3il
✅ Added get_killed_action_space_priority() method
✅ Added get_best_action() method with argmax functionality
✅ Added BestMoveSelector class for clean interface

📊 KEY FEATURES:
- Calculates how many valid spots each move would kill
- Returns priority grid: -k for valid moves (k = spots killed), -inf for invalid
- argmax gives best move (kills fewest spots)
- Efficient implementation using numba-compiled functions

🧪 VERIFIED EXAMPLES:
1. 3x3 test case matches expected pattern
2. 5x5 automated gameplay works correctly
3. BestMoveSelector provides clean interface

🎮 USAGE EXAMPLE:
analyzer = KillSpaceAnalyzer((n, n), args)
priority_grid = analyzer.get_killed_action_space_priority(state)
best_move = analyzer.get_best_action(state)
# OR use the enhanced selector:
selector = BestMoveSelector((n, n), args)
best_move = selector.select_best_move(state)

✨ The implementation is ready to use!
\n=